<a href="https://colab.research.google.com/github/MohHaroon/XAI-based-ZSL-for-IIDS/blob/master/DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Streamlit Demo -
Explainable Zero-Shot Attack Attribution in Software-Defined Industrial Networks

## Load CatBoost model and data splits -
### SDN Dataset

In [1]:
!pip install catboost

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Reloading CatBoost
from catboost import CatBoostClassifier
loaded_cat = CatBoostClassifier()
loaded_cat.load_model('/content/drive/MyDrive/IRP_Models/catboost_sdn_20260123_040438.cbm')

CatBoostClassifier(depth=5, iterations=100, learning_rate=1, loss_function='Logloss', verbose=0)

In [4]:
import os
import joblib
import pandas as pd

base_path = '/content/drive/MyDrive/IRP_Models/'
split_path = os.path.join(base_path, 'Data_Splits/')

X_train = joblib.load(os.path.join(split_path, f'X_train.pkl'))
X_test = joblib.load(os.path.join(split_path, f'X_test.pkl'))
y_train = joblib.load(os.path.join(split_path, f'y_train.pkl'))
y_test = joblib.load(os.path.join(split_path, f'y_test.pkl'))


# Adding the column names
feature_names = ["proto_number", "Dur", "Mean", "Stddev", "Min", "Max", "Pkts", "Bytes",
                 "Spkts", "Dpkts", "Sbytes", "Dbytes", "Srate", "Drate", "Sum",
                 "TnBPSrcIP", "TnBPDstIP", "TnP_PSrcIP", "TnP_PDstIP", "TnP_PerProto",
                 "TnP_Per_Dport", "N_IN_Conn_P_DstIP", "N_IN_Conn_P_SrcIP"]

X_test = pd.DataFrame(X_test, columns=feature_names)
X_train = pd.DataFrame(X_train, columns=feature_names)

y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f"\nSuccessfully loaded Data Splits for {len(X_test)} test packets.")


Successfully loaded Data Splits for 42000 test packets.


## Zero-Shot Attribution Layer (ZSAL)

In [5]:
import shap
explainer = shap.TreeExplainer(loaded_cat)
def generate_zasl_label(shap_values, feature_names):

    positive_contributions = [(val, name) for val, name in zip(shap_values, feature_names) if val > 0]
    # Sort by impact (highest value first)
    sorted_contributions = sorted(positive_contributions, key=lambda x: x[0], reverse=True)

    # Extract top features
    top_features = [item[1] for item in sorted_contributions[:3]]
    high_impact_extra = [item[1] for item in sorted_contributions[3:] if item[0] > 2]

    final_feature_list = top_features + high_impact_extra

    if not top_features:
        return "Generic Anomaly"

    return f"{'-'.join(final_feature_list)}-based-Attribution"



## SHAP Overlap Automation Model (SOAM)

## Load SHAP profiles

In [6]:
soam_path = '/content/drive/MyDrive/IRP_Models/SOAM_Baselines2/'

# Loading back for the final similarity check
tp_mean = joblib.load(f'{soam_path}tp_mean_profile_20260325_115545.pkl')
fp_mean = joblib.load(f'{soam_path}fp_mean_profile_20260325_115545.pkl')
normal_mean = joblib.load(f'{soam_path}normal_mean_profile_20260325_115545.pkl')
fn_mean = joblib.load(f'{soam_path}fn_mean_profile_20260325_115545.pkl')

### Prediction Confidence and Verdict

In [7]:
def calculate_weighted_soam(instance_shap, mean_profile):
    from scipy.spatial.distance import cosine
    from scipy.stats import spearmanr
    import numpy as np

    # 1. Cosine Similarity (Weight: 50%)
    cos_sim = 1 - cosine(instance_shap, mean_profile)

    # 2. Spearman Rank Correlation (Weight: 25%)
    spearman_corr, _ = spearmanr(instance_shap, mean_profile)

    # Calculate absolute differences for each feature
    absolute_diffs = np.abs(instance_shap - mean_profile)
    # Calculate the mean of these differences
    mad_score = np.mean(absolute_diffs)

    # 3. Jaccard-style Sign Overlap (Weight: 25%)
    instance_signs = np.sign(instance_shap)
    mean_signs = np.sign(mean_profile)
    sign_overlap = (instance_signs == mean_signs).mean()

    # 4. Final Weighted Aggregation
    weighted_score = (0.70 * cos_sim) + (0.1 * spearman_corr) + (0.1 * sign_overlap) + (0.1 * mad_score)


    return {
        "Cosine": cos_sim,
        "Spearman": spearman_corr,
        "Sign_Overlap": sign_overlap,
        "MAD": mad_score,
        "Weighted_Reliability": weighted_score
    }

In [8]:
def get_soam_verdict(instance_shap, tp_mean, fp_mean):


    # Calculate scores for both reference groups
    tp_scores = calculate_weighted_soam(instance_shap, tp_mean)
    #print("Cosine, SPearman, Sign_Overlap, MAD for tp", tp_scores['Cosine'], tp_scores['Spearman'], tp_scores['Sign_Overlap'],tp_scores['MAD'])

    fp_scores = calculate_weighted_soam(instance_shap, fp_mean)
    #print("Cosine, SPearman, Sign_Overlap, MAD for fp", fp_scores['Cosine'], fp_scores['Spearman'], fp_scores['Sign_Overlap'], fp_scores['MAD'])

    tp_rel = tp_scores['Weighted_Reliability']
    fp_rel = fp_scores['Weighted_Reliability']

    # Logic to determine the final system action
    if tp_rel > fp_rel and (tp_rel-fp_rel) > 0.15:
        verdict = f"CONFIRMED ATTACK"
        action = "Display in System"
        "SUSPECT Attack (Potential False Negative)", "MEDIUM - For Analyst Review"
    elif fp_rel > tp_rel:
        verdict = f"SUSPECT Attack (Potential False Positive)"
        action = "LOW - For Analyst Review"
    else:
        verdict = "SUSPECT Attack (Potential True Positive)"
        action = "HIGH - For Analyst Review"

    return {
        "Verdict": verdict,
        "Action": action,
        "TrueProfile_Similarity": tp_rel,
        "FalseProfile_Similarity": fp_rel
    }

In [9]:
def get_soam_verdict_NormalValidation(instance_shap, normal_mean, fp_mean):
    # Calculate similarity to Normal baseline
    normal_sim = calculate_weighted_soam(instance_shap, normal_mean)['Weighted_Reliability']
    # Calculate similarity to False Positive (Missed Attack) baseline
    fn_sim = calculate_weighted_soam(instance_shap, fn_mean)['Weighted_Reliability']
    #print(f"Similarity to Normal Baseline: {normal_sim:.4f}")
    #print(f"Similarity to False Positive (Missed Attack) Baseline: {fn_sim:.4f}")

    if fn_sim > normal_sim and fn_sim > 0.4:
        #target_shap = instance_shap[0]
        label = generate_zasl_label(instance_shap, X_test.columns.tolist())
        verdict = "SUSPECT Attack (Potential False Negative)"
        action = "MEDIUM - For Analyst Review"
        return {
                "Label": label,
                "Verdict": verdict,
                "Action": action,
                "TrueProfile_Similarity": normal_sim,
                "FalseProfile_Similarity": fn_sim
            }
    label = "Normal Traffic"
    verdict = "NORMAL Traffic"
    action = "No Action Required"
    return {
                "Label": label,
                "Verdict": verdict,
                "Action": action,
                "TrueProfile_Similarity": normal_sim,
                "FalseProfile_Similarity": fn_sim
            }

## CLI Simulation

In [10]:
import time
import pandas as pd
import numpy as np
from datetime import datetime

def run_sdin_cli_simulation(X_test, y_test, model, explainer, profiles, num_packets=10):
    """
    Simulates a live CLI monitoring tool for an Industrial SDN.
    """
    print("="*80)
    print(f" SDIN SECURITY MONITOR - X-IDS PROTOCOLE | START TIME: {datetime.now().strftime('%H:%M:%S')}")
    print("="*80)
    print(f"{'TIMESTAMP':<12} | {'RESULT':<8} | {'CONFIDENCE':<10} | {'VERDICT/LABEL'}")
    print("-"*80)

    # Randomly sample packets to simulate live traffic
    samples = X_test.sample(num_packets)

    for i, (idx, row) in enumerate(samples.iterrows()):
        packet = row.values.reshape(1, -1)
        packet_df = pd.DataFrame([row], columns=X_test.columns)

        # 1. Model Prediction
        pred = model.predict(packet_df)[0]

        # 2. XAI Processing
        shap_vals = explainer.shap_values(packet_df)[0]

        timestamp = datetime.now().strftime('%H:%M:%S')

        if pred == 1:
            # ATTACK PATH
            label = generate_zasl_label(shap_vals, X_test.columns.tolist())
            verdict = get_soam_verdict(shap_vals, profiles['tp_mean'], profiles['fp_mean'])

            # Clean up verdict string for CLI display
            status = f"\033[91m{verdict['Verdict']}\033[0m" # Red Text
            if "LOW" in verdict['Action']:
                status = f"\033[93m{verdict['Verdict']}\033[0m"


            conf = f"{verdict['TrueProfile_Similarity']:.2%}"
            details = f"[{label}] -> {verdict['Verdict']}, {verdict["Action"]}"
        else:
            # NORMAL PATH
            # Simplified integrity check for CLI
            normal_sim = get_soam_verdict_NormalValidation(shap_vals, profiles['normal_mean'], profiles['fn_mean'])

            status = f"\033[92m{normal_sim['Verdict']}\033[0m" # Green Text

            if "MEDIUM" in normal_sim['Action']:
                status = f"\033[38;5;208m{normal_sim['Verdict']}\033[0m" # Orange
           # Yellow Text
            # if normal_sim['Verdict'] == "SUSPECT Attack (Potential False Negative)":
            #     label = normal_sim['Label']
            conf = f"{normal_sim['FalseProfile_Similarity']:.2%}"
            details = f"[{normal_sim["Label"]}] -> {normal_sim['Verdict']} , {normal_sim["Action"]}"

        print(f"{timestamp:<12} | {status:<17} | {conf:<10} | {details}")

        # Simulate network delay
        time.sleep(0.8)

    print("-"*80)
    print("SIMULATION COMPLETE | ALL PACKETS PROCESSED")
    print("="*80)


In [11]:
# Setup Profiles for the CLI
profile_store = {
    'tp_mean': tp_mean,
    'fp_mean': fp_mean,
    'normal_mean': normal_mean,
    'fn_mean' : fn_mean
}
run_sdin_cli_simulation(X_test, y_test, loaded_cat, explainer, profile_store, num_packets=15)

 SDIN SECURITY MONITOR - X-IDS PROTOCOLE | START TIME: 12:05:05
TIMESTAMP    | RESULT   | CONFIDENCE | VERDICT/LABEL
--------------------------------------------------------------------------------
12:05:05     | SUSPECT Attack (Potential False Positive) | 50.64%     | [Dbytes-Sum-Spkts-based-Attribution] -> SUSPECT Attack (Potential False Positive), LOW - For Analyst Review
12:05:05     | CONFIRMED ATTACK | 34.98%     | [proto_number-Spkts-Dpkts-based-Attribution] -> CONFIRMED ATTACK, Display in System
12:05:06     | SUSPECT Attack (Potential False Positive) | 52.12%     | [TnP_PerProto-Spkts-Dbytes-based-Attribution] -> SUSPECT Attack (Potential False Positive), LOW - For Analyst Review
12:05:07     | SUSPECT Attack (Potential False Positive) | 53.19%     | [Dbytes-Spkts-N_IN_Conn_P_DstIP-based-Attribution] -> SUSPECT Attack (Potential False Positive), LOW - For Analyst Review
12:05:08     | SUSPECT Attack (Potential True Positive) | 24.15%     | [Sbytes-Drate-Dbytes-based-Attributio